# Analiza statystyczna wyników

Notatnik przeprowadza weryfikację istotności różnic pomiędzy badanymi modelami
oraz analizę zgodności ich predykcji. Operuje wyłącznie na zapisanych plikach
wynikowych, nie wymaga ponownego uczenia modeli ani dostępu do zbioru danych.

## Uzasadnienie przyjętej metody

Zbiór testowy obejmuje 1142 nagrania pochodzące od zaledwie 14 mówców. Nagrania
tej samej osoby nie stanowią obserwacji niezależnych: skuteczność modelu dla
danego głosu przekłada się jednocześnie na wszystkie jego wypowiedzi. Testy
zakładające niezależność poszczególnych obserwacji, takie jak standardowy test
McNemara, prowadziłyby zatem do zawyżenia istotności statystycznej.

Zastosowano **bootstrap blokowy na poziomie mówców**. W każdej replikacji losowana
jest ze zwracaniem czternastoelementowa próba mówców, a badana miara wyznaczana
jest na wszystkich nagraniach wylosowanych osób. Jednostką losowania jest mówca,
nie zaś pojedyncze nagranie, dzięki czemu zachowana zostaje wewnętrzna korelacja
obserwacji.

Ponieważ każdy model uczono pięciokrotnie przy różnych ziarnach inicjalizacji,
w każdej replikacji miarę wyznacza się niezależnie dla wszystkich przebiegów,
a porównaniu podlega różnica wartości uśrednionych. Procedura uwzględnia zatem
obydwa źródła zmienności: dobór mówców oraz stochastyczny charakter optymalizacji.
Nie wymaga też arbitralnego wskazania jednego przebiegu jako reprezentatywnego.

Przy zaledwie czternastu klastrach precyzja wyznaczonych przedziałów pozostaje
ograniczona, a formułowane na ich podstawie wnioski wymagają ostrożności.

## Wymagane pliki

```
results/
├── m1_mfcc_5seeds.json
├── m2_magnitude_5seeds.json
├── m3_reim_5seeds.json
├── m4_complex_5seeds.json
└── m4_stable_5seeds.json     (opcjonalnie)
```

## 1. Konfiguracja

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

RESULTS = Path("../results")

N_CLASSES = 6
EMOTIONS = ["ANG", "DIS", "FEA", "HAP", "NEU", "SAD"]
LABELS_PL = ["złość", "wstręt", "strach", "radość", "neutralny", "smutek"]

MODELE = {
    "M1 (MFCC)":      "m1_mfcc_5seeds.json",
    "M2 (|STFT|)":    "m2_magnitude_5seeds.json",
    "M3 (Re/Im)":     "m3_reim_5seeds.json",
    "M4 (zespolony)": "m4_complex_5seeds.json",
}

N_REPLIKACJI = 10_000
POZIOM_UFNOSCI = 0.95
ZIARNO_BOOTSTRAP = 42
HIERARCHICZNY = True

D = {}
for nazwa, plik in MODELE.items():
    with open(RESULTS / plik, encoding="utf-8") as f:
        D[nazwa] = json.load(f)
    print(f"{nazwa:16s} {len(D[nazwa]['runs'])} przebiegów, "
          f"ziarna {D[nazwa]['seeds']}")

M1 (MFCC)        5 przebiegów, ziarna [42, 43, 44, 45, 46]
M2 (|STFT|)      5 przebiegów, ziarna [42, 43, 44, 45, 46]
M3 (Re/Im)       5 przebiegów, ziarna [42, 43, 44, 45, 46]
M4 (zespolony)   5 przebiegów, ziarna [42, 43, 44, 45, 46]


## 2. Struktura zbioru testowego

Identyfikator mówcy stanowi pierwszy człon nazwy pliku, na przykład dla nagrania
`1003_DFA_ANG_XX` jest to `1003`. Podział na bloki odpowiada zatem podziałowi
zbioru testowego na mówców.

In [3]:
# etykiety rzeczywiste sa identyczne dla wszystkich modeli i przebiegow
y = np.array(D["M1 (MFCC)"]["runs"][0]["true_utterance"])

for nazwa, d in D.items():
    for r in d["runs"]:
        assert np.array_equal(np.array(r["true_utterance"]), y), \
            f"{nazwa}: niezgodny zbiór testowy"
print(f"kontrola: wszystkie modele oceniane na tym samym zbiorze "
      f"({len(y)} nagrań)")

# identyfikatory mowcow
sciezka_id = RESULTS / "test_ids.txt"
if sciezka_id.exists():
    ids = [l.strip() for l in open(sciezka_id, encoding="utf-8") if l.strip()]
else:
    ids = D["M1 (MFCC)"]["runs"][0].get("file_ids")
    if ids is None:
        raise FileNotFoundError(
            "Brak identyfikatorów nagrań. Wymagany plik results/test_ids.txt "
            "albo pole 'file_ids' w wynikach.")

mowcy = np.array([i.split("_")[0] for i in ids])
unikalni = np.unique(mowcy)

assert len(ids) == len(y), "liczba identyfikatorów niezgodna z liczbą etykiet"
print(f"mówców w zbiorze testowym: {len(unikalni)}")
print(f"nagrań na mówcę: od {np.bincount(np.unique(mowcy, return_inverse=True)[1]).min()} "
      f"do {np.bincount(np.unique(mowcy, return_inverse=True)[1]).max()}")

kontrola: wszystkie modele oceniane na tym samym zbiorze (1142 nagrań)
mówców w zbiorze testowym: 14
nagrań na mówcę: od 76 do 82


## 3. Miara UAR

In [4]:
def uar(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    # usredniona czulosc klasowa, bez wazenia licznoscia klas
    suma = 0.0
    for k in range(N_CLASSES):
        maska = y_true == k
        if maska.any():
            suma += (y_pred[maska] == k).mean()
    return suma / N_CLASSES


# predykcje wszystkich przebiegow, ksztalt [liczba_przebiegow, liczba_nagran]
P = {nazwa: np.array([r["pred_utterance"] for r in d["runs"]])
     for nazwa, d in D.items()}

print(f"{'model':18s}{'UAR (średnia)':>16s}{'odch. std':>12s}")
for nazwa, pred in P.items():
    v = np.array([uar(y, p) for p in pred])
    print(f"{nazwa:18s}{v.mean():>16.4f}{v.std():>12.4f}")

model                UAR (średnia)   odch. std
M1 (MFCC)                   0.6117      0.0149
M2 (|STFT|)                 0.6295      0.0095
M3 (Re/Im)                  0.6001      0.0053
M4 (zespolony)              0.4609      0.0208


## 4. Bootstrap blokowy

Indeksy replikacji wyznaczane są jednokrotnie i wykorzystywane we wszystkich
porównaniach, dzięki czemu różnice pomiędzy parami modeli są wzajemnie
porównywalne.

In [5]:
rng = np.random.default_rng(ZIARNO_BOOTSTRAP)

N_PRZEBIEGOW = len(P["M1 (MFCC)"])
bloki = {m: np.where(mowcy == m)[0] for m in unikalni}

# poziom 1: mowcy (blok = wszystkie nagrania jednej osoby)
# poziom 2: przebiegi uczenia, te same indeksy dla obu porownywanych modeli,
#           dzieki czemu zachowane zostaje parowanie po ziarnie
indeksy, przebiegi = [], []
for _ in range(N_REPLIKACJI):
    wybrani = rng.choice(unikalni, len(unikalni), replace=True)
    indeksy.append(np.concatenate([bloki[m] for m in wybrani]))
    przebiegi.append(rng.integers(0, N_PRZEBIEGOW, size=N_PRZEBIEGOW))

print(f"wygenerowano {len(indeksy)} replikacji")
print(f"srednia licznosc proby: {np.mean([len(i) for i in indeksy]):.0f} nagran")
print(f"poziom drugi: {'wlaczony' if HIERARCHICZNY else 'wylaczony'} "
      f"({N_PRZEBIEGOW} przebiegow)")


def porownaj(model_a: str, model_b: str) -> dict:
    """Roznica UAR miedzy modelami, przedzial ufnosci i wartosc p.

    Bootstrap dwupoziomowy: losowanie mowcow ze zwracaniem oraz, przy
    HIERARCHICZNY=True, losowanie przebiegow uczenia ze zwracaniem.
    Wartosc p wyznaczana jest jako dwustronny odsetek replikacji po
    przeciwnej stronie zera.
    """
    obserwowana = (np.mean([uar(y, p) for p in P[model_b]])
                   - np.mean([uar(y, p) for p in P[model_a]]))

    roznice = np.empty(N_REPLIKACJI)
    for i, idx in enumerate(indeksy):
        y_b = y[idx]
        if HIERARCHICZNY:
            pa = P[model_a][przebiegi[i]]
            pb = P[model_b][przebiegi[i]]
        else:
            pa, pb = P[model_a], P[model_b]
        roznice[i] = (np.mean([uar(y_b, p[idx]) for p in pb])
                      - np.mean([uar(y_b, p[idx]) for p in pa]))

    alfa = (1 - POZIOM_UFNOSCI) / 2
    lo, hi = np.percentile(roznice, [100 * alfa, 100 * (1 - alfa)])
    p_wart = min(1.0, 2 * min((roznice <= 0).mean(), (roznice >= 0).mean()))

    return {"delta": obserwowana, "lo": lo, "hi": hi, "p": p_wart,
            "istotna": not (lo <= 0 <= hi)}

wygenerowano 10000 replikacji
srednia licznosc proby: 1142 nagran
poziom drugi: wlaczony (5 przebiegow)


## 5. Porównanie kolejnych wariantów potoku

Zestawienie czterech wariantów potoku. Tylko para M2–M3 różni się pojedynczym elementem (liczbą kanałów wejściowych). Przejście M3 → M4 obejmuje jednocześnie zmianę arytmetyki, szerokości warstw, normalizacji, aktywacji, sposobu redukcji, komórki rekurencyjnej, inicjalizacji i rozmiaru partii.

In [6]:
KROKI = [
    ("M1 (MFCC)",      "M2 (|STFT|)",    "parametryzacja widma i stopnie redukcji"),
    ("M2 (|STFT|)",    "M3 (Re/Im)",     "udostępnienie informacji fazowej"),
    ("M3 (Re/Im)",     "M4 (zespolony)", "potok zespolony (arytmetyka i pozostałe zmiany)"),
    ("M1 (MFCC)",      "M4 (zespolony)", "wszystkie zmiany łącznie"),
]

wiersze = []
for a, b, czynnik in KROKI:
    w = porownaj(a, b)
    wiersze.append({
        "porównanie": f"{a.split()[0]} → {b.split()[0]}",
        "badany czynnik": czynnik,
        "ΔUAR [p.p.]": round(100 * w["delta"], 2),
        "95% CI": f"[{100*w['lo']:+.2f}; {100*w['hi']:+.2f}]",
        "p": f"{w['p']:.4f}",
        "istotna": "tak" if w["istotna"] else "nie",
    })

tabela = pd.DataFrame(wiersze)
display(tabela)

,porównanie,badany czynnik,ΔUAR [p.p.],95% CI,p,istotna
0,M1 → M2,parametryzacja widma i stopnie redukcji,1.78,[-1.37; +4.65],0.2486,nie
1,M2 → M3,udostępnienie informacji fazowej,-2.93,[-5.10; -0.78],0.0110,tak
2,M3 → M4,potok zespolony (arytmetyka i pozostałe zmiany),-13.93,[-17.45; -10.90],0.0000,tak
3,M1 → M4,wszystkie zmiany łącznie,-15.09,[-18.08; -11.99],0.0000,tak


### Postać do wklejenia w tekście pracy

In [7]:
print(r"\begin{tabular}{llccc}")
print(r"\hline")
print(r"\textbf{Porównanie} & \textbf{Główna zmiana} & "
      r"\textbf{$\Delta$UAR} & \textbf{95\% CI} & \textbf{$p$} \\")
print(r"\hline")
for a, b, czynnik in KROKI:
    w = porownaj(a, b)
    delta = f"${100*w['delta']:+.2f}$".replace(".", "{,}")
    ci = f"$[{100*w['lo']:+.2f};\\ {100*w['hi']:+.2f}]$".replace(".", "{,}")
    pw = (r"$<0{,}001$" if w["p"] < 0.001
          else f"${w['p']:.3f}$".replace(".", "{,}"))
    print(f"{a.split()[0]} $\\rightarrow$ {b.split()[0]} & {czynnik} & "
          f"{delta} & {ci} & {pw} \\\\")
print(r"\hline")
print(r"\end{tabular}")

\begin{tabular}{llccc}
\hline
\textbf{Porównanie} & \textbf{Główna zmiana} & \textbf{$\Delta$UAR} & \textbf{95\% CI} & \textbf{$p$} \\
\hline
M1 $\rightarrow$ M2 & parametryzacja widma i stopnie redukcji & $+1{,}78$ & $[-1{,}37;\ +4{,}65]$ & $0{,}249$ \\
M2 $\rightarrow$ M3 & udostępnienie informacji fazowej & $-2{,}93$ & $[-5{,}10;\ -0{,}78]$ & $0{,}011$ \\
M3 $\rightarrow$ M4 & potok zespolony (arytmetyka i pozostałe zmiany) & $-13{,}93$ & $[-17{,}45;\ -10{,}90]$ & $<0{,}001$ \\
M1 $\rightarrow$ M4 & wszystkie zmiany łącznie & $-15{,}09$ & $[-18{,}08;\ -11{,}99]$ & $<0{,}001$ \\
\hline
\end{tabular}


## 6. Konfiguracja stabilizowana

Wariant stabilizowany modelu zespolonego stanowi odrębny eksperyment
optymalizacyjny. Odmienna procedura uczenia uniemożliwia zestawienie go
z modelami rzeczywistymi, dopuszczalne jest natomiast porównanie z konfiguracją
podstawową tego samego modelu.

In [8]:
sciezka_stab = RESULTS / "m4_stable_5seeds.json"

if sciezka_stab.exists():
    with open(sciezka_stab, encoding="utf-8") as f:
        D["M4 (stabilizowany)"] = json.load(f)
    P["M4 (stabilizowany)"] = np.array(
        [r["pred_utterance"] for r in D["M4 (stabilizowany)"]["runs"]])

    v = np.array([uar(y, p) for p in P["M4 (stabilizowany)"]])
    print(f"M4 stabilizowany: UAR {v.mean():.4f} ± {v.std():.4f}\n")

    w = porownaj("M4 (zespolony)", "M4 (stabilizowany)")
    print(f"różnica UAR      : {100*w['delta']:+.2f} p.p.")
    print(f"95% CI           : [{100*w['lo']:+.2f}; {100*w['hi']:+.2f}]")
    print(f"wartość p        : {w['p']:.4f}")
    print(f"istotna          : {'tak' if w['istotna'] else 'nie'}")
else:
    print("Brak pliku z wynikami konfiguracji stabilizowanej.")

M4 stabilizowany: UAR 0.4679 ± 0.0142

różnica UAR      : +0.71 p.p.
95% CI           : [-2.81; +4.11]
wartość p        : 0.6870
istotna          : nie


## 7. Rozrzut wyników pomiędzy przebiegami

Porównanie stabilności procesu uczenia. Analiza ma charakter opisowy: przy pięciu
powtórzeniach możliwe jest wskazanie rzędu wielkości rozrzutu, nie przeprowadzono
natomiast testu odnoszącego się bezpośrednio do różnicy wariancji.

In [9]:
wiersze = []
for nazwa, pred in P.items():
    v = np.array([uar(y, p) for p in pred])
    wiersze.append({
        "model": nazwa,
        "wartości UAR": "  ".join(f"{x:.4f}" for x in v),
        "średnia": round(v.mean(), 4),
        "odch. std": round(v.std(), 4),
        "rozstęp [p.p.]": round(100 * (v.max() - v.min()), 2),
    })

display(pd.DataFrame(wiersze))

,model,wartości UAR,średnia,odch. std,rozstęp [p.p.]
0,M1 (MFCC),0.6019 0.5997 0.6028 0.6395 0.6146,0.6117,0.0149,3.99
1,M2 (|STFT|),0.6237 0.6208 0.6230 0.6337 0.6462,0.6295,0.0095,2.55
2,M3 (Re/Im),0.6095 0.5950 0.6024 0.5974 0.5965,0.6001,0.0053,1.45
3,M4 (zespolony),0.4263 0.4693 0.4492 0.4844 0.4751,0.4609,0.0208,5.81
4,M4 (stabilizowany),0.4848 0.4824 0.4659 0.4471 0.4595,0.4679,0.0142,3.78


## 8. Zgodność predykcji modeli M1 oraz M4

Porównanie predykcji skrajnych ogniw drabiny ablacyjnej na poszczególnych
nagraniach. Wartości uśredniono po wszystkich parach przebiegów obydwu modeli.

Zestawienie opisuje komplementarność błędów konkretnych modeli i nie pozwala
wnioskować o zawartości informacyjnej samych reprezentacji: z pełnej transformaty
STFT można wyznaczyć zarówno jej moduł, jak i współczynniki mel-cepstralne, wobec
czego reprezentacja mel-cepstralna stanowi funkcję reprezentacji zespolonej.

In [10]:
pr, pc = P["M1 (MFCC)"], P["M4 (zespolony)"]
n_par = len(pr) * len(pc)
n = len(y)

kategorie = np.zeros(4)
zgodnosc = 0.0
for a in pr:
    for b in pc:
        ok_a, ok_b = (a == y), (b == y)
        zgodnosc += (a == b).mean()
        kategorie += [(ok_a & ok_b).sum(), (ok_a & ~ok_b).sum(),
                      (~ok_a & ok_b).sum(), (~ok_a & ~ok_b).sum()]
kategorie /= n_par
zgodnosc /= n_par

opisy = ["obydwa modele poprawnie", "poprawnie tylko M1",
         "poprawnie tylko M4", "obydwa modele błędnie"]

display(pd.DataFrame({
    "sytuacja": opisy,
    "liczba nagrań": np.round(kategorie, 1),
    "udział": [f"{100*k/n:.1f}%" for k in kategorie],
}))

print(f"\nzgodność predykcji, niezależnie od poprawności: {100*zgodnosc:.1f}%")

,sytuacja,liczba nagrań,udział
0,obydwa modele poprawnie,424.2,37.1%
1,poprawnie tylko M1,274.2,24.0%
2,poprawnie tylko M4,103.0,9.0%
3,obydwa modele błędnie,340.6,29.8%



zgodność predykcji, niezależnie od poprawności: 53.3%


In [11]:
# rozklad rozbieznosci w podziale na klasy rzeczywiste
tab = np.zeros((N_CLASSES, 4))
for a in pr:
    for b in pc:
        ok_a, ok_b = (a == y), (b == y)
        for k in range(N_CLASSES):
            m = y == k
            tab[k] += [(ok_a & ~ok_b & m).sum(), (~ok_a & ok_b & m).sum(),
                       (ok_a & ok_b & m).sum(), (~ok_a & ~ok_b & m).sum()]
tab /= n_par

display(pd.DataFrame(np.round(tab, 1), index=LABELS_PL,
                     columns=["tylko M1", "tylko M4",
                              "oba poprawnie", "oba błędnie"]))

,tylko M1,tylko M4,oba poprawnie,oba błędnie
złość,23.4,10.0,151.4,10.2
wstręt,65.5,12.5,36.5,80.5
strach,60.1,12.1,33.1,89.7
radość,51.7,26.9,69.7,46.7
neutralny,45.7,14.1,57.5,49.7
smutek,27.8,27.4,76.0,63.8


## 9. Zapis wyników

In [12]:
wynik = {
    "metoda": "bootstrap blokowy na poziomie mówców",
    "liczba_replikacji": N_REPLIKACJI,
    "poziom_ufnosci": POZIOM_UFNOSCI,
    "ziarno": ZIARNO_BOOTSTRAP,
    "liczba_mowcow": int(len(unikalni)),
    "liczba_nagran": int(n),
    "porownania": {},
}

for a, b, czynnik in KROKI:
    w = porownaj(a, b)
    wynik["porownania"][f"{a} -> {b}"] = {
        "czynnik": czynnik,
        "delta_uar": round(w["delta"], 6),
        "ci_low": round(w["lo"], 6),
        "ci_high": round(w["hi"], 6),
        "p": round(w["p"], 6),
        "istotna": w["istotna"],
    }

if "M4 (stabilizowany)" in P:
    w = porownaj("M4 (zespolony)", "M4 (stabilizowany)")
    wynik["porownania"]["M4 -> M4 stabilizowany"] = {
        "czynnik": "mechanizmy stabilizujące proces uczenia",
        "delta_uar": round(w["delta"], 6),
        "ci_low": round(w["lo"], 6),
        "ci_high": round(w["hi"], 6),
        "p": round(w["p"], 6),
        "istotna": w["istotna"],
    }

with open(RESULTS / "analiza_statystyczna.json", "w", encoding="utf-8") as f:
    json.dump(wynik, f, indent=2, ensure_ascii=False)

print(f"zapisano: {RESULTS / 'analiza_statystyczna.json'}")

zapisano: ..\results\analiza_statystyczna.json
